<a href="https://colab.research.google.com/github/aranavkumar/Veloura/blob/main/08_Veloura_Master_Coordinator_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PHASE 8 — CELL 1: MASTER COORDINATOR INITIALIZATION
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONNECT TO GOOGLE DRIVE
# ------------------------------------------------------------

PROJECT_CANDIDATES = [
    "/content/drive_phase7/MyDrive/Veloura_Project",
    "/content/drive/MyDrive/Veloura_Project"
]

PROJECT_DIR = next(
    (
        path for path in PROJECT_CANDIDATES
        if os.path.isdir(path)
    ),
    None
)

if PROJECT_DIR is None:
    from google.colab import drive
    drive.mount("/content/drive_phase8")
    PROJECT_DIR = (
        "/content/drive_phase8/MyDrive/Veloura_Project"
    )

assert os.path.isdir(PROJECT_DIR), (
    f"Project directory not found: {PROJECT_DIR}"
)

print("PROJECT DIRECTORY:", PROJECT_DIR)


# ------------------------------------------------------------
# 2. DEFINE INPUT FILES
# ------------------------------------------------------------

PHASE5_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "agent_logs",
    "lesion_analysis_sample_logs.json"
)

PHASE6_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "reasoning_context",
    "reasoning_context_sample_logs.json"
)

PHASE7_SAFETY_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "safety_agent",
    "safety_agent_sample_logs.json"
)

PHASE7_COORDINATOR_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "safety_agent",
    "coordinator_integration_sample_logs.json"
)

# ------------------------------------------------------------
# 3. VERIFY INPUT FILES
# ------------------------------------------------------------

input_files = {
    "Phase 5 - Lesion Analysis": PHASE5_JSON,
    "Phase 6 - Reasoning Context": PHASE6_JSON,
    "Phase 7 - Safety Agent": PHASE7_SAFETY_JSON,
    "Phase 7 - Coordinator Integration": PHASE7_COORDINATOR_JSON
}

print("\nINPUT FILE CHECK")
print("=" * 65)

for name, path in input_files.items():
    exists = os.path.isfile(path)
    print(
        f"[{'FOUND' if exists else 'MISSING'}] "
        f"{name}: {os.path.basename(path)}"
    )

assert all(
    os.path.isfile(path)
    for path in input_files.values()
), "One or more required Phase 5–7 files are missing."


# ------------------------------------------------------------
# 4. LOAD ALL AGENT OUTPUTS
# ------------------------------------------------------------

with open(PHASE5_JSON, "r") as file:
    phase5_data = json.load(file)

with open(PHASE6_JSON, "r") as file:
    phase6_data = json.load(file)

with open(PHASE7_SAFETY_JSON, "r") as file:
    phase7_safety_data = json.load(file)

with open(PHASE7_COORDINATOR_JSON, "r") as file:
    phase7_coordinator_data = json.load(file)


# ------------------------------------------------------------
# 5. BASIC DATA VALIDATION
# ------------------------------------------------------------

datasets = {
    "Phase 5": phase5_data,
    "Phase 6": phase6_data,
    "Phase 7 Safety": phase7_safety_data,
    "Phase 7 Coordinator": phase7_coordinator_data
}

for name, data in datasets.items():
    assert isinstance(data, list), (
        f"{name} output must be a list."
    )

    print(f"{name} records loaded:", len(data))


# Confirm image IDs are consistent across all phases
image_id_sets = {
    name: {record["image_id"] for record in data}
    for name, data in datasets.items()
}

reference_ids = image_id_sets["Phase 5"]

assert all(
    ids == reference_ids
    for ids in image_id_sets.values()
), "Image IDs do not match across agent outputs."

assert len(reference_ids) == 6, (
    "Expected six sample images."
)

# ------------------------------------------------------------
# 6. CREATE OUTPUT DIRECTORIES
# ------------------------------------------------------------

MASTER_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "master_coordinator"
)

MASTER_REPORT_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "master_coordinator"
)

os.makedirs(MASTER_OUTPUT_DIR, exist_ok=True)
os.makedirs(MASTER_REPORT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 7. INITIALIZATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("VELOURA MASTER COORDINATOR INITIALIZATION")
print("=" * 65)

print("Total unique images:", len(reference_ids))
print("Phase 5 records:", len(phase5_data))
print("Phase 6 records:", len(phase6_data))
print("Phase 7 safety records:", len(phase7_safety_data))
print("Phase 7 coordinator records:", len(phase7_coordinator_data))

print("\nImage IDs:", sorted(reference_ids))

print("\nMaster output directory:", MASTER_OUTPUT_DIR)
print("Master report directory:", MASTER_REPORT_DIR)

print("\nSUCCESS: MASTER COORDINATOR INITIALIZED!")

Mounted at /content/drive_phase8
PROJECT DIRECTORY: /content/drive_phase8/MyDrive/Veloura_Project

INPUT FILE CHECK
[FOUND] Phase 5 - Lesion Analysis: lesion_analysis_sample_logs.json
[FOUND] Phase 6 - Reasoning Context: reasoning_context_sample_logs.json
[FOUND] Phase 7 - Safety Agent: safety_agent_sample_logs.json
[FOUND] Phase 7 - Coordinator Integration: coordinator_integration_sample_logs.json
Phase 5 records loaded: 6
Phase 6 records loaded: 6
Phase 7 Safety records loaded: 6
Phase 7 Coordinator records loaded: 6

VELOURA MASTER COORDINATOR INITIALIZATION
Total unique images: 6
Phase 5 records: 6
Phase 6 records: 6
Phase 7 safety records: 6
Phase 7 coordinator records: 6

Image IDs: ['ISIC_0025481', 'ISIC_0028145', 'ISIC_0031157', 'ISIC_0031437', 'ISIC_0033038', 'ISIC_0034119']

Master output directory: /content/drive_phase8/MyDrive/Veloura_Project/outputs/master_coordinator
Master report directory: /content/drive_phase8/MyDrive/Veloura_Project/reports/master_coordinator

SUCCESS

In [ ]:
# ============================================================
# PHASE 8 — CELL 2: MASTER ORCHESTRATION ENGINE
# ============================================================

from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. INDEX UPSTREAM AGENT RESULTS
# ------------------------------------------------------------

phase5_lookup = {
    record["image_id"]: record
    for record in phase5_data
}

phase6_lookup = {
    record["image_id"]: record
    for record in phase6_data
}

safety_lookup = {
    record["image_id"]: record
    for record in phase7_safety_data
}

coordinator_lookup = {
    record["image_id"]: record
    for record in phase7_coordinator_data
}


# ------------------------------------------------------------
# 2. MASTER ORCHESTRATION FUNCTION
# ------------------------------------------------------------

def orchestrate_case(image_id):

    lesion = phase5_lookup[image_id]
    context = phase6_lookup[image_id]
    safety = safety_lookup[image_id]
    coordinator = coordinator_lookup[image_id]

    # --------------------------------------------------------
    # Validate upstream records
    # --------------------------------------------------------

    required_agents = {
        "lesion_analysis": lesion,
        "reasoning_context": context,
        "safety_agent": safety,
        "phase7_coordinator": coordinator
    }

    agent_availability = {
        name: record is not None
        for name, record in required_agents.items()
    }

    all_agents_available = all(
        agent_availability.values()
    )

    # --------------------------------------------------------
    # Validate safety restrictions
    # --------------------------------------------------------

    safety_gate_passed = (
        safety.get("safety_decision")
        == "NO_AUTOMATED_CLINICAL_CLEARANCE"
        and coordinator.get("safety_gate", {}).get("passed")
        is True
    )

    human_review_required = (
        safety.get("safety_decision")
        == "HUMAN_REVIEW_REQUIRED"
        or coordinator.get("workflow_status")
        == "BLOCKED_PENDING_HUMAN_REVIEW"
    )

    # --------------------------------------------------------
    # Determine workflow status
    # --------------------------------------------------------

    if not all_agents_available:

        workflow_status = "INCOMPLETE_AGENT_OUTPUTS"
        next_action = "REPAIR_MISSING_AGENT_OUTPUTS"

    elif human_review_required or not safety_gate_passed:

        workflow_status = "BLOCKED_PENDING_HUMAN_REVIEW"
        next_action = "ESCALATE_TO_CLINICIAN"

    else:

        # This prototype does not authorize clinical clearance.
        workflow_status = "REQUIRES_AUTHORIZED_REVIEW"
        next_action = "HUMAN_REVIEW_REQUIRED"

    # --------------------------------------------------------
    # Build unified evidence record
    # --------------------------------------------------------

    master_record = {

        "image_id": image_id,

        "agent": "Veloura Master Coordinator",

        "orchestration_status": workflow_status,

        "agent_status": {
            "lesion_analysis": lesion.get("status"),
            "reasoning_context": context.get(
                "context_completeness", {}
            ).get("status"),
            "safety_agent": safety.get("safety_decision"),
            "phase7_coordinator": coordinator.get(
                "workflow_status"
            )
        },

        "agent_availability": agent_availability,

        "evidence_summary": {
            "lesion_analysis_status": lesion.get("status"),
            "segmentation_method": lesion.get(
                "segmentation_method"
            ),
            "feature_status": context.get(
                "lesion_context", {}
            ).get("feature_status"),
            "missing_information": context.get(
                "missing_information", []
            ),
            "context_completeness": context.get(
                "context_completeness", {}
            )
        },

        "safety_summary": {
            "safety_decision": safety.get(
                "safety_decision"
            ),
            "safety_flags": safety.get(
                "safety_flags", []
            ),
            "safety_gate_passed": safety_gate_passed,
            "human_review_required": human_review_required
        },

        "workflow_permissions": {
            "automatic_acceptance": False,
            "independent_diagnosis": False,
            "automated_clinical_clearance": False,
            "human_review_required": True
        },

        "coordinator_action": next_action,

        "clinical_authority": {
            "autonomous_diagnosis": False,
            "autonomous_treatment_recommendation": False,
            "final_decision_owner": "QUALIFIED_CLINICIAN"
        },

        "orchestrated_at": datetime.now(
            timezone.utc
        ).isoformat()
    }

    return master_record


# ------------------------------------------------------------
# 3. ORCHESTRATE ALL CASES
# ------------------------------------------------------------

all_image_ids = sorted(reference_ids)

master_results = [
    orchestrate_case(image_id)
    for image_id in all_image_ids
]


# ------------------------------------------------------------
# 4. DISPLAY MASTER WORKFLOW
# ------------------------------------------------------------

print("=" * 75)
print("VELOURA MASTER COORDINATOR — ORCHESTRATION RESULTS")
print("=" * 75)

for record in master_results:

    print("\nIMAGE:", record["image_id"])

    print(
        "Orchestration status:",
        record["orchestration_status"]
    )

    print(
        "Coordinator action:",
        record["coordinator_action"]
    )

    print(
        "Safety decision:",
        record["safety_summary"]["safety_decision"]
    )

    print(
        "All agents available:",
        all(record["agent_availability"].values())
    )

    print(
        "Automatic acceptance:",
        record["workflow_permissions"]["automatic_acceptance"]
    )


# ------------------------------------------------------------
# 5. MASTER STATUS SUMMARY
# ------------------------------------------------------------

master_status_df = pd.DataFrame([

    {
        "image_id": record["image_id"],
        "orchestration_status": record[
            "orchestration_status"
        ],
        "coordinator_action": record[
            "coordinator_action"
        ],
        "all_agents_available": all(
            record["agent_availability"].values()
        ),
        "safety_gate_passed": record[
            "safety_summary"
        ]["safety_gate_passed"],
        "human_review_required": record[
            "workflow_permissions"
        ]["human_review_required"]
    }

    for record in master_results

])

print("\n" + "=" * 75)
print("MASTER COORDINATOR SUMMARY")
print("=" * 75)

print(master_status_df.to_string(index=False))

print("\nTotal cases orchestrated:", len(master_results))

print("\nSUCCESS: MASTER ORCHESTRATION ENGINE EXECUTED!")

VELOURA MASTER COORDINATOR — ORCHESTRATION RESULTS

IMAGE: ISIC_0025481
Orchestration status: BLOCKED_PENDING_HUMAN_REVIEW
Coordinator action: ESCALATE_TO_CLINICIAN
Safety decision: HUMAN_REVIEW_REQUIRED
All agents available: True
Automatic acceptance: False

IMAGE: ISIC_0028145
Orchestration status: BLOCKED_PENDING_HUMAN_REVIEW
Coordinator action: ESCALATE_TO_CLINICIAN
Safety decision: HUMAN_REVIEW_REQUIRED
All agents available: True
Automatic acceptance: False

IMAGE: ISIC_0031157
Orchestration status: BLOCKED_PENDING_HUMAN_REVIEW
Coordinator action: ESCALATE_TO_CLINICIAN
Safety decision: HUMAN_REVIEW_REQUIRED
All agents available: True
Automatic acceptance: False

IMAGE: ISIC_0031437
Orchestration status: BLOCKED_PENDING_HUMAN_REVIEW
Coordinator action: ESCALATE_TO_CLINICIAN
Safety decision: HUMAN_REVIEW_REQUIRED
All agents available: True
Automatic acceptance: False

IMAGE: ISIC_0033038
Orchestration status: BLOCKED_PENDING_HUMAN_REVIEW
Coordinator action: ESCALATE_TO_CLINICIAN
Saf

In [ ]:
# ============================================================
# PHASE 8 — CELL 3: MASTER EXPORT & FINAL VERIFICATION
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# 1. EXPORT MASTER COORDINATOR JSON
# ------------------------------------------------------------

MASTER_JSON = os.path.join(
    MASTER_OUTPUT_DIR,
    "master_coordinator_sample_logs.json"
)

with open(MASTER_JSON, "w") as file:
    json.dump(master_results, file, indent=4)


# ------------------------------------------------------------
# 2. EXPORT MASTER SUMMARY CSV
# ------------------------------------------------------------

MASTER_CSV = os.path.join(
    MASTER_REPORT_DIR,
    "master_coordinator_sample_summary.csv"
)

master_status_df.to_csv(MASTER_CSV, index=False)


# ------------------------------------------------------------
# 3. EXPORT AGENT STATUS MATRIX
# ------------------------------------------------------------

agent_matrix = pd.DataFrame([

    {
        "image_id": record["image_id"],
        "lesion_analysis": record["agent_status"][
            "lesion_analysis"
        ],
        "reasoning_context": record["agent_status"][
            "reasoning_context"
        ],
        "safety_agent": record["agent_status"][
            "safety_agent"
        ],
        "phase7_coordinator": record["agent_status"][
            "phase7_coordinator"
        ],
        "master_coordinator": record[
            "orchestration_status"
        ]
    }

    for record in master_results

])

AGENT_MATRIX_CSV = os.path.join(
    MASTER_REPORT_DIR,
    "master_agent_status_matrix.csv"
)

agent_matrix.to_csv(AGENT_MATRIX_CSV, index=False)


# ------------------------------------------------------------
# 4. RELOAD EXPORTED FILES
# ------------------------------------------------------------

with open(MASTER_JSON, "r") as file:
    exported_master = json.load(file)

exported_summary = pd.read_csv(MASTER_CSV)
exported_matrix = pd.read_csv(AGENT_MATRIX_CSV)


# ------------------------------------------------------------
# 5. FINAL VERIFICATION TESTS
# ------------------------------------------------------------

expected_ids = {
    record["image_id"]
    for record in phase5_data
}

tests = {

    "Six master records exported":
        len(exported_master) == 6,

    "Six master summary rows exported":
        len(exported_summary) == 6,

    "Six agent matrix rows exported":
        len(exported_matrix) == 6,

    "Master image IDs match upstream data":
        {
            record["image_id"]
            for record in exported_master
        } == expected_ids,

    "All upstream agents available":
        all(
            all(record["agent_availability"].values())
            for record in exported_master
        ),

    "All cases blocked pending human review":
        all(
            record["orchestration_status"]
            == "BLOCKED_PENDING_HUMAN_REVIEW"
            for record in exported_master
        ),

    "All cases escalated to clinician":
        all(
            record["coordinator_action"]
            == "ESCALATE_TO_CLINICIAN"
            for record in exported_master
        ),

    "Safety gate remains closed":
        all(
            record["safety_summary"]["safety_gate_passed"]
            is False
            for record in exported_master
        ),

    "Human review remains mandatory":
        all(
            record["workflow_permissions"][
                "human_review_required"
            ] is True
            for record in exported_master
        ),

    "Automatic acceptance remains disabled":
        all(
            record["workflow_permissions"][
                "automatic_acceptance"
            ] is False
            for record in exported_master
        ),

    "Independent diagnosis remains disabled":
        all(
            record["workflow_permissions"][
                "independent_diagnosis"
            ] is False
            for record in exported_master
        ),

    "Automated clinical clearance remains disabled":
        all(
            record["workflow_permissions"][
                "automated_clinical_clearance"
            ] is False
            for record in exported_master
        ),

    "Final decision owner is qualified clinician":
        all(
            record["clinical_authority"][
                "final_decision_owner"
            ] == "QUALIFIED_CLINICIAN"
            for record in exported_master
        )
}


# ------------------------------------------------------------
# 6. DISPLAY VERIFICATION RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("VELOURA PHASE 8 — FINAL VERIFICATION")
print("=" * 75)

for test_name, passed in tests.items():

    print(
        f"[{'PASS' if passed else 'FAIL'}] {test_name}"
    )

passed_count = sum(tests.values())
total_count = len(tests)

print("\nVerification:", passed_count, "/", total_count)


# ------------------------------------------------------------
# 7. DISPLAY EXPORTED FILES
# ------------------------------------------------------------

print("\nEXPORTED FILES")

print("Master JSON:", MASTER_JSON)
print("Master Summary CSV:", MASTER_CSV)
print("Agent Status Matrix:", AGENT_MATRIX_CSV)

print("\nAGENT STATUS MATRIX")
print(agent_matrix.to_string(index=False))


# ------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------

if all(tests.values()):

    print("\nSUCCESS: PHASE 8 COMPLETED AND VERIFIED!")

else:

    print("\nWARNING: SOME VERIFICATION TESTS FAILED.")

VELOURA PHASE 8 — FINAL VERIFICATION
[PASS] Six master records exported
[PASS] Six master summary rows exported
[PASS] Six agent matrix rows exported
[PASS] Master image IDs match upstream data
[PASS] All upstream agents available
[PASS] All cases blocked pending human review
[PASS] All cases escalated to clinician
[PASS] Safety gate remains closed
[PASS] Human review remains mandatory
[PASS] Automatic acceptance remains disabled
[PASS] Independent diagnosis remains disabled
[PASS] Automated clinical clearance remains disabled
[PASS] Final decision owner is qualified clinician

Verification: 13 / 13

EXPORTED FILES
Master JSON: /content/drive_phase8/MyDrive/Veloura_Project/outputs/master_coordinator/master_coordinator_sample_logs.json
Master Summary CSV: /content/drive_phase8/MyDrive/Veloura_Project/reports/master_coordinator/master_coordinator_sample_summary.csv
Agent Status Matrix: /content/drive_phase8/MyDrive/Veloura_Project/reports/master_coordinator/master_agent_status_matrix.csv